# Algorithme de Shor pour N = 15

$$N = 15 = 3 \times 5$$

Factorisation via recherche de période:
$$a^r \equiv 1 \pmod N \quad\Longrightarrow\quad \gcd(a^{r/2} \pm 1, N)$$

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from math import gcd
import matplotlib.pyplot as plt

### Exponentiation modulaire

Pour $a=2$, $N=15$:
$$U|x\rangle|y\rangle = |x\rangle|y \cdot 2^x \bmod 15\rangle$$

La période $r$ satisfait $2^r \equiv 1 \pmod{15}$, donc $r=4$.

In [ ]:
def c_amod15(a, power):
    U = QuantumCircuit(4)
    for _ in range(power):
        if a == 2:
            U.swap(2, 3)
            U.swap(1, 2)
            U.swap(0, 1)
        elif a == 7:
            U.swap(0, 1)
            U.swap(1, 2)
            U.swap(2, 3)
        elif a == 11:
            U.swap(0, 3)
            U.swap(1, 2)
        elif a == 13:
            U.swap(0, 2)
            U.swap(1, 3)
        elif a == 4:
            for q in range(4):
                U.x(q)
    U = U.to_gate()
    U.name = f'{a}^{power} mod 15'
    return U.control()

In [ ]:
def shor_circuit_n15(a=2, n_count=8):
    q_count = QuantumRegister(n_count, 'count')
    q_target = QuantumRegister(4, 'target')
    c = ClassicalRegister(n_count, 'c')
    qc = QuantumCircuit(q_count, q_target, c)

    qc.x(0)
    qc.h(q_count)

    for i in range(n_count):
        qc.append(c_amod15(a, 2**i), [q_count[i]] + q_target[:])

    qc.append(qft_circuit(n_count).inverse(), q_count)
    qc.measure(q_count, c)
    return qc

def qft_circuit(n):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            qc.cp(np.pi / 2**(j - i), j, i)
    return qc

In [ ]:
n_count = 8
qc_shor = shor_circuit_n15(2, n_count)
backend = AerSimulator()
qc_t = transpile(qc_shor, backend)
result = backend.run(qc_t, shots=2048).result()
counts = result.get_counts()
plot_histogram(counts, title='Shor N=15, a=2: distribution des phases')

### Fractions continues et période

À partir de la mesure $m$, on approxime:
$$\frac{m}{2^{n\_\text{count}}} \approx \frac{s}{r}$$

via l'algorithme des fractions continues pour retrouver $r$.

In [ ]:
def continued_fractions(frac, max_denom=100):
    conv = []
    a = int(np.floor(frac))
    conv.append(a)
    frac = frac - a
    if frac == 0:
        return conv
    for _ in range(max_denom):
        frac = 1 / frac
        a = int(np.floor(frac))
        conv.append(a)
        frac = frac - a
        if frac == 0:
            break
    return conv

def convergents_from_cf(cf):
    n0, n1 = 0, 1
    d0, d1 = 1, 0
    convs = []
    for a in cf:
        n2 = a * n1 + n0
        d2 = a * d1 + d0
        convs.append((n2, d2))
        n0, n1 = n1, n2
        d0, d1 = d1, d2
    return convs

In [ ]:
def find_period(measured_phase, n_count, a, N):
    frac = measured_phase / 2**n_count
    cf = continued_fractions(frac)
    convs = convergents_from_cf(cf)
    for s, r in convs:
        if r > 0 and pow(a, r, N) == 1:
            return r
    return None

a = 2
N = 15
periods_found = []
for bitstr, count in counts.items():
    measured = int(bitstr, 2)
    r = find_period(measured, n_count, a, N)
    if r is not None:
        periods_found.append(r)

print('Périodes trouvées:', set(periods_found))
if periods_found:
    r = max(set(periods_found), key=periods_found.count)
    print(f'Période la plus fréquente: r = {r}')
    print(f'Facteurs: {gcd(a**(r//2) - 1, N)}, {gcd(a**(r//2) + 1, N)}')

### Analyse de probabilité de succès

La probabilité de succès dépend:
- Du nombre $n$ de qubits de comptage
- De la base $a$ choisie
- De la précision des fractions continues

$$P_{\text{succès}} \geq 1 - \frac{1}{2^{n - \lceil \log_2 r \rceil - 1}}$$

In [ ]:
def success_probability(a, N, n_count, n_trials=100):
    success = 0
    for _ in range(n_trials):
        qc = shor_circuit_n15(a, n_count)
        qc_t = transpile(qc, backend)
        counts = backend.run(qc_t, shots=1).result().get_counts()
        bitstr = list(counts.keys())[0]
        measured = int(bitstr, 2)
        r = find_period(measured, n_count, a, N)
        if r is not None and r % 2 == 0 and a**(r//2) % N != N - 1:
            success += 1
    return success / n_trials

for a_test in [2, 7, 11, 13]:
    p = success_probability(a_test, 15, 8, 50)
    print(f'a={a_test}: P_succès = {p:.2f}')

## Questions

**Q1.** Tester l'algorithme avec $a=4$. Expliquer pourquoi la période trouvée est $r=2$ et comment cela affecte le calcul des facteurs. Dans quels cas $a^{r/2} \equiv -1 \pmod N$ et que faire alors?

**Q2.** Faire varier le nombre de qubits de comptage $n_{\text{count}}$ de 4 à 10 et tracer la probabilité de succès en fonction de $n_{\text{count}}$. À partir de combien de qubits la probabilité dépasse-t-elle 90%?

In [ ]:
# Q2: Probabilité de succès vs n_count
n_range = range(4, 11)
probs = []
for n in n_range:
    p = success_probability(2, 15, n, 100)
    probs.append(p)

plt.plot(list(n_range), probs, 'o-', linewidth=2)
plt.axhline(0.9, color='r', linestyle='--', label='90%')
plt.xlabel('Qubits de comptage n')
plt.ylabel('Probabilité de succès')
plt.title('Succès de Shor N=15 vs n_count')
plt.grid(True)
plt.legend()
plt.show()